# Entrenamiento de CPIA-recommender en Colab (H8)

Ejecuta las celdas **en orden**. Guía completa: `docs/colab.md`. Antes: activa la GPU (Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU).

**Parámetros.** Elige la muestra (`dev` para probar, `completo` para entrenar de verdad).

In [ ]:
MUESTRA = "completo"  # "dev" o "completo"
CARPETA_DRIVE = "/content/drive/MyDrive/CPIA"
REPO_URL = "https://github.com/ArnauMontull/CPIA-recommender.git"
REPO_DIR = "/content/CPIA-recommender"

**1. GPU.** Comprueba que Colab te ha dado una GPU. Si sale error o `False`: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU, y vuelve a empezar.

In [ ]:
!nvidia-smi
import torch
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("AVISO: sin GPU. Activa la T4 en Entorno de ejecución > Cambiar tipo de entorno de ejecución.")

**2. Drive.** Monta tu Google Drive (te pedirá permiso en una ventana emergente).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**3. Repositorio.** Clona el repo (o hace `git pull` si ya existe). Si es privado, usa el secreto `GITHUB_TOKEN` de Colab; el token nunca se imprime.

In [ ]:
import os, subprocess

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    # Nunca mostramos la URL con el token: filtramos la salida.
    salida = (r.stdout + r.stderr)
    if token:
        salida = salida.replace(token, "***")
    print(salida.strip())
    if r.returncode != 0:
        raise RuntimeError("git ha fallado (mira el mensaje de arriba)")

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None  # repo público o sin secreto

url = REPO_URL
if token:
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    git("pull", "--ff-only", url, "main", cwd=REPO_DIR)
else:
    git("clone", url, REPO_DIR)
    if token:  # que el token no se quede guardado en .git/config
        git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)
os.chdir(REPO_DIR)
print("Directorio de trabajo:", os.getcwd())

**4. Dependencias.** Instala `requirements.txt` quitando la línea de torch (Colab ya trae torch con CUDA).

In [ ]:
import re
lineas = open("requirements.txt", encoding="utf-8").read().splitlines()
sin_torch = [l for l in lineas if not re.match(r"\s*torch\b", l)]
open("/content/requirements_colab.txt", "w", encoding="utf-8").write("\n".join(sin_torch) + "\n")
!pip install -q -r /content/requirements_colab.txt
import torch
print("torch de Colab:", torch.__version__)

**5. Datos.** Copia `datos_<muestra>.zip` desde Drive, verifica el sha256 (si existe el `.sha256`) y descomprime en el repo.

In [ ]:
import hashlib, os, shutil, zipfile

nombre = f"datos_{MUESTRA}"
zip_drive = f"{CARPETA_DRIVE}/{nombre}.zip"
sha_drive = f"{CARPETA_DRIVE}/{nombre}.sha256"
assert os.path.exists(zip_drive), f"No encuentro {zip_drive}. Súbelo a Drive (ver docs/colab.md)."

zip_local = f"/content/{nombre}.zip"
shutil.copy(zip_drive, zip_local)

if os.path.exists(sha_drive):
    esperado = open(sha_drive, encoding="utf-8").read().split()[0].lower()
    h = hashlib.sha256()
    with open(zip_local, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    assert h.hexdigest() == esperado, "sha256 distinto: el zip está corrupto o es de otra versión. Vuelve a subirlo."
    print("sha256 correcto:", esperado)
else:
    print("No hay .sha256 en Drive: salto la verificación.")

with zipfile.ZipFile(zip_local) as z:
    z.extractall(REPO_DIR)
print("Descomprimido en", REPO_DIR)
!ls data/processed

**6. Tests.** Comprueba que el código funciona en este entorno antes de gastar GPU.

In [ ]:
!python -m pytest tests -q

**7. Entrenamiento.** Los checkpoints se guardan directamente en Drive (`.../modelos/<muestra>_tau<τ>_s<semilla>/`). Si la sesión se corta, vuelve a ejecutar esta celda: reanuda desde `ultimo.pt`. Para el barrido de τ (T1) cambia `TAU` a 0.02, 0.05 y 0.1 y ejecuta cada vez.

In [ ]:
TAU = 0.05  # T1: barrido en val con 0.02, 0.05, 0.1
!python -m src.entrenar --muestra {MUESTRA} --device cuda --tau {TAU} --salida /content/drive/MyDrive/CPIA/modelos

**8. Copia final a Drive.** Por si el entrenamiento dejó checkpoints en el disco local de Colab, cópialos a Drive (el disco local se borra al cerrarse la sesión).

In [ ]:
import glob, os, shutil
destino = f"{CARPETA_DRIVE}/modelos"
os.makedirs(destino, exist_ok=True)
locales = glob.glob(f"{REPO_DIR}/modelos/**/*", recursive=True) + glob.glob(f"{REPO_DIR}/checkpoints/**/*", recursive=True)
n = 0
for p in locales:
    if os.path.isfile(p):
        rel = os.path.relpath(p, REPO_DIR)
        d = os.path.join(destino, rel)
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copy2(p, d)
        n += 1
print(f"{n} archivos copiados a {destino}" if n else "Nada que copiar (los checkpoints ya están en Drive).")